# 03 - Alternative Features & Scorecard Modeling

**Track 3 Lead**: Open / In Progress  
**Course**: UC Berkeley DATASCI 207 (Machine Learning at Scale)  
**Target Deliverable**: Project Milestone & Final Submission  
**Scope**: Feature cohort engineering, thin-file customer segmentation, baseline logistic scorecard benchmarks, and gradient boosting (LightGBM) tuning.

---

### Environment & Data Ingestion (Google Colab)
Run the cell below if executing in Google Colab to mount Google Drive and pull dataset files to `/content/local_data`.


In [ ]:
try:
    from google.colab import drive
    import os
    import shutil

    # Mount Google Drive only if not already mounted
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')

    # Source and destination paths
    source_dataset = "/content/drive/MyDrive/DATASCI207_finalproject"
    destination_path = "/content/local_data"

    def copy_dataset_clean(src, dst):
        """Recursively copy files while skipping Google Docs/Sheets shortcuts and existing files."""
        for root, dirs, files in os.walk(src):
            rel_path = os.path.relpath(root, src)
            dest_dir = os.path.join(dst, rel_path)
            os.makedirs(dest_dir, exist_ok=True)

            for f in files:
                # Skip Google Docs, Sheets, Slides shortcuts and hidden files
                if f.endswith(('.gdoc', '.gsheet', '.gslides')) or f.startswith('.'):
                    continue

                src_file = os.path.join(root, f)
                dst_file = os.path.join(dest_dir, f)

                # Only copy if file doesn't exist locally or size has changed
                if not os.path.exists(dst_file) or os.path.getsize(src_file) != os.path.getsize(dst_file):
                    try:
                        shutil.copy2(src_file, dst_file)
                    except Exception as e:
                        print(f"Skipping {f}: {e}")

    # Verify path and sync
    if os.path.exists(source_dataset):
        print("Syncing dataset files from Google Drive to /content/local_data...")
        copy_dataset_clean(source_dataset, destination_path)
        print("Sync complete! Available in /content/local_data:", os.listdir(destination_path))
    else:
        print(f"Directory not found: {source_dataset}")
        print("Contents of /content/drive/MyDrive:")
        try:
            print(os.listdir("/content/drive/MyDrive"))
        except FileNotFoundError:
            print("MyDrive folder not found in /content/drive.")
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")


### Cleaned Static Baseline Ingestion (Task 1)

Loads the cleaned and engineered static applicant dataset (`application_train_cleaned.parquet`) produced by Task 1.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Path resolution: checks Colab runtime first, falls back to local repo
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path('/content/local_data') if Path('/content/local_data').exists() else PROJECT_ROOT / 'data'
PROCESSED_PATH = DATA_DIR / 'processed' / 'application_train_cleaned.parquet'

# Ingest cleaned static baseline from Task 1
print(f"Data root: {DATA_DIR}")
if PROCESSED_PATH.exists():
    df_cleaned = pd.read_parquet(PROCESSED_PATH)
    print(f"Loaded cleaned data: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]:,} columns")
else:
    print(f"File not found: {PROCESSED_PATH}")
    print("Ensure application_train_cleaned.parquet is uploaded to Google Drive under DATASCI207_finalproject/processed/")
